# Module 08: The Distillation Idea

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, you'll build intuition for knowledge distillation using
simple, synthetic examples. No GPU needed, no model training — just
visualizations that show why soft labels carry more information than hard
labels, and how temperature controls that information flow.

## Setup

We only need NumPy and Matplotlib. Nothing heavy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Class names for our synthetic 5-class problem
CLASS_NAMES = [
    "Billing\nDispute",
    "Payment\nIssue",
    "Account\nMgmt",
    "Credit\nReport",
    "Fraud",
]

## Hard labels vs. soft labels

Imagine a customer complaint that's been labeled "Billing Dispute" in our
dataset. The **hard label** is a one-hot vector — 100% billing dispute,
0% everything else.

Now imagine running that same complaint through a big, accurate teacher
model. The teacher doesn't just say "billing dispute." It says something
like "70% billing dispute, 20% payment issue, 5% account management..."
— that's a **soft label**, a full probability distribution.

Let's visualize both.

In [ ]:
# Hard label: one-hot encoding for "Billing Dispute"
hard_label = np.array([1.0, 0.0, 0.0, 0.0, 0.0])

# Soft label: what a teacher model might predict for the same example
soft_label = np.array([0.70, 0.20, 0.05, 0.03, 0.02])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hard label
axes[0].bar(CLASS_NAMES, hard_label, color="steelblue", edgecolor="black")
axes[0].set_title("Hard Label (from dataset)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Probability")
axes[0].set_ylim(0, 1.1)
for i, v in enumerate(hard_label):
    axes[0].text(i, v + 0.03, f"{v:.0%}", ha="center", fontsize=10)

# Soft label
axes[1].bar(CLASS_NAMES, soft_label, color="coral", edgecolor="black")
axes[1].set_title("Soft Label (from teacher model)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Probability")
axes[1].set_ylim(0, 1.1)
for i, v in enumerate(soft_label):
    axes[1].text(i, v + 0.03, f"{v:.0%}", ha="center", fontsize=10)

fig.suptitle(
    'Same complaint: "I was charged twice on my credit card statement"',
    fontsize=12,
    style="italic",
    y=1.02,
)
plt.tight_layout()
plt.show()

## What the soft label tells us

Look at the soft label carefully:

- **Billing Dispute** (70%) and **Payment Issue** (20%) are closely
  related — the teacher sees significant similarity between them.
- **Account Management** (5%) has a small but nonzero probability —
  there's a loose connection.
- **Credit Report** (3%) and **Fraud** (2%) are barely relevant.

The hard label throws all of this away. Every wrong class is equally
wrong (0%). The soft label preserves the **structure** — which classes
are similar to which. This extra information is what Geoff Hinton called
**"dark knowledge."**

## Temperature scaling

When a model computes its output probabilities, it uses the **softmax**
function on its raw output values (called **logits**). We can introduce
a **temperature** parameter that controls how "peaked" or "spread out"
the resulting distribution is:

- **T = 1** (default): normal softmax, often very confident
- **T > 1**: softer, more spread out — reveals class relationships
- **T → ∞**: approaches uniform distribution

Let's see this in action.

In [ ]:
def softmax_with_temperature(logits, temperature):
    """Apply softmax with temperature scaling."""
    scaled = logits / temperature
    # Subtract max for numerical stability
    scaled = scaled - np.max(scaled)
    exp_scaled = np.exp(scaled)
    return exp_scaled / np.sum(exp_scaled)


# Synthetic logits — raw model output before softmax
# These represent the teacher's "confidence" for each class
logits = np.array([5.0, 3.0, 1.0, 0.5, 0.2])

temperatures = [1, 2, 5, 10]

print("Logits (raw model output):", logits)
print()

for T in temperatures:
    probs = softmax_with_temperature(logits, T)
    print(f"T = {T:2d}: ", end="")
    for name, p in zip(CLASS_NAMES, probs):
        print(f"{name.replace(chr(10), ' ')} = {p:.3f}  ", end="")
    print()

## Visualizing the temperature effect

Let's plot the probability distributions at each temperature side by
side. Watch how the distribution spreads out as temperature increases.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

for idx, (T, color) in enumerate(zip(temperatures, colors)):
    probs = softmax_with_temperature(logits, T)
    axes[idx].bar(CLASS_NAMES, probs, color=color, edgecolor="black", alpha=0.85)
    axes[idx].set_title(f"Temperature = {T}", fontsize=13, fontweight="bold")
    axes[idx].set_ylim(0, 0.85)

    for i, v in enumerate(probs):
        axes[idx].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

    if idx == 0:
        axes[idx].set_ylabel("Probability")

fig.suptitle(
    "Same logits, different temperatures — watch the distribution spread",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

## Reading the temperature plots

At **T = 1**, the model is very confident: "Billing Dispute" dominates
and the other classes are almost invisible. The dark knowledge is
hidden — you can barely see that "Payment Issue" is the second choice.

At **T = 5** and **T = 10**, the distribution has spread out. Now you
can clearly see:
- "Billing Dispute" and "Payment Issue" are the top two — **they're related**
- "Account Management" is in the middle — **loosely connected**
- "Credit Report" and "Fraud" are at the bottom — **not related**

This is the information that transfers to the student. The student
doesn't just learn "this is billing dispute" — it learns the
**similarity structure** of the label space.

## Quantifying information content

We can measure information content using **entropy**. Higher entropy
means more information about class relationships (less certainty, more
nuance in the distribution).

In [ ]:
def entropy(probs):
    """Compute entropy of a probability distribution (in bits)."""
    # Filter out zeros to avoid log(0)
    p = probs[probs > 0]
    return max(0.0, -np.sum(p * np.log2(p)))  # avoid -0.0 display


print("Information content (entropy in bits):\n")
print(f"  Hard label (one-hot):    {entropy(hard_label):.3f} bits")
print()

for T in temperatures:
    probs = softmax_with_temperature(logits, T)
    print(f"  Soft label at T = {T:2d}:    {entropy(probs):.3f} bits")

max_entropy = np.log2(len(CLASS_NAMES))
print(f"\n  Maximum possible:        {max_entropy:.3f} bits (uniform distribution)")
print(
    "\nHigher entropy = more information about class relationships"
    "\n(but too high means the signal is washed out)"
)

## Connection to the course

In **Week 6**, your teacher will be a larger model (395M parameters).
Your student will be the model you've been training all semester (149M).
The teacher's soft predictions on training data become additional
training signal.

The process:
1. Run the teacher on every training example to get soft labels
2. Train the student on a **blend** of hard labels (ground truth) and
   soft labels (teacher knowledge)
3. The student inherits some of the teacher's understanding without
   getting any bigger or slower

The result: a small model that punches above its weight, because it
learned from a more knowledgeable teacher — not just from the raw data.

## Check yourself

Before moving on, make sure you can answer these:

1. **What is the difference between a hard label and a soft label?**
   A hard label is a one-hot vector (100% one class, 0% everything
   else). A soft label is a probability distribution from the teacher
   model that assigns some probability to every class.

2. **Why do soft labels carry more information than hard labels?**
   Because they encode relationships between classes. The teacher's
   probabilities tell you which wrong answers are "close" to the right
   answer — information that hard labels throw away.

3. **What does temperature do?**
   Higher temperature makes the probability distribution more spread
   out (softer), which reveals more information about class
   relationships. Lower temperature makes it more peaked (harder).

4. **What is "dark knowledge"?**
   The information hidden in the teacher's near-misses — the
   probabilities assigned to wrong classes that reveal the structure
   of the problem.

5. **In Week 6, what will the teacher and student be?**
   The teacher is a 395M parameter model. The student is the 149M
   parameter model you've been building all semester.